<h2 align='center'>Codebasics ML Course: ML Flow Dagshub Tutorial</h2>

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100]))

In [3]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

#### Handle class imbalance

In [4]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)
np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619]))

### Track Experiments

In [5]:
models = [
    (
        "Logistic Regression", 
        {"C": 1, "solver": 'liblinear'},
        LogisticRegression(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        {"n_estimators": 30, "max_depth": 3},
        RandomForestClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [6]:
reports = []

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

In [7]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

In [9]:
# dagshub setup

import dagshub
dagshub.init(repo_owner='BigM19', repo_name='mlflow_dagshub', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=961bbcac-2adb-4fd5-aedf-56bffbbb348f&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=144b3e281a8f16a345aa3ec179413e11a2e184cb8ed9be6ad8d3761ea2c7ad11




Accessing as BigM19

Initialized MLflow to track repo "BigM19/mlflow_dagshub"

Repository BigM19/mlflow_dagshub initialized!

In [ ]:
# Initialize MLflow
mlflow.set_tracking_uri("https://dagshub.com/BigM19/mlflow_dagshub.mlflow")
mlflow.set_experiment("Anomaly Detection with Params")


for i, element in enumerate(models):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):        
        mlflow.log_params(params)
        mlflow.log_metrics({
            'accuracy': report['accuracy'],
            'recall_class_1': report['1']['recall'],
            'recall_class_0': report['0']['recall'],
            'f1_score_macro': report['macro avg']['f1-score']
        })  
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, name="model")
        else:
            mlflow.sklearn.log_model(model, name="model")  

2026/01/28 14:48:50 INFO mlflow.tracking.fluent: Experiment with name 'Anomaly Detection with Params' does not exist. Creating a new experiment.
2026/01/28 14:48:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/28 14:49:04 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Logistic Regression at: https://dagshub.com/BigM19/mlflow_dagshub.mlflow/#/experiments/0/runs/15e513a03eac4ebb923f72136dbba19d
🧪 View experiment at: https://dagshub.com/BigM19/mlflow_dagshub.mlflow/#/experiments/0


2026/01/28 14:49:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/28 14:49:24 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Random Forest at: https://dagshub.com/BigM19/mlflow_dagshub.mlflow/#/experiments/0/runs/172865e3f18347a0a11c433d74584ecf
🧪 View experiment at: https://dagshub.com/BigM19/mlflow_dagshub.mlflow/#/experiments/0


2026/01/28 14:49:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/28 14:49:48 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run XGBClassifier at: https://dagshub.com/BigM19/mlflow_dagshub.mlflow/#/experiments/0/runs/5777b1c14e814bd3ab56126c6792804c
🧪 View experiment at: https://dagshub.com/BigM19/mlflow_dagshub.mlflow/#/experiments/0


2026/01/28 14:49:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/28 14:50:11 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run XGBClassifier With SMOTE at: https://dagshub.com/BigM19/mlflow_dagshub.mlflow/#/experiments/0/runs/50c75c18b1bd4cd7929485afb17f9d83
🧪 View experiment at: https://dagshub.com/BigM19/mlflow_dagshub.mlflow/#/experiments/0
